# Lab 04 · Small agentic patterns

**Prepared by Mustafa Assaf** · Beginner level · 60–90 minute session

Start with the small worked example, read the expected output, then change one thing.
Use a CPU runtime and run the cells from top to bottom. No repository clone is needed.

**Practice mode:** model answers are explicitly scripted so you can inspect the code
without a key. They demonstrate control flow, not model quality. MCP, A2A and LangGraph
examples run their actual libraries locally even in practice mode; their setup cells
install pinned packages if needed.

**Optional live calls:** add `FREEINFERENCE_API_KEY` in Colab's Secrets panel, allow
this notebook access, and change `MODE` to `"live"`. The initial model is
`qwen3.6-35b` on FreeInference; `BASE_URL`, `MODEL` and `SECRET_NAME` can be changed
for another compatible provider. Keep the key out of cells and outputs.

### Shared setup
The next cell contains the reusable model client. Run it as supplied; the short teaching examples follow below.

In [ ]:
"""Small, provider-configurable teaching client. Standard library only."""
import getpass
import json
import threading
import time
import urllib.error
import urllib.parse
import urllib.request

# Shared by every Client in this notebook/process, including parallel workflows.
_NETWORK_SLOTS = threading.BoundedSemaphore(2)


class _NoRedirect(urllib.request.HTTPRedirectHandler):
    def redirect_request(self, req, fp, code, msg, headers, newurl):
        return None  # Never forward a credential to another host.


class Client:
    """Practice responses are explicit fixtures, not an LLM or model simulation."""
    def __init__(self, mode="practice", model="", family="", base_url="", api_key="",
                 max_calls=24, max_tokens=512, timeout=45, token_field="max_tokens", extra_body=None):
        if extra_body is not None and (not isinstance(extra_body, dict) or
                set(extra_body) & {"model", "messages", "max_tokens", "max_completion_tokens", "stream"}):
            raise ValueError("extra_body must be a dict and cannot override the request contract or budgets")
        if mode not in {"practice", "live"}:
            raise ValueError("mode must be practice or live")
        if token_field not in {"max_tokens", "max_completion_tokens"}:
            raise ValueError("Check which output-token field your provider supports")
        if max_calls < 1 or max_tokens < 1 or timeout <= 0:
            raise ValueError("Call, output, and timeout bounds must be positive")
        if mode == "live":
            url = urllib.parse.urlsplit(base_url)
            if url.scheme != "https" or not url.netloc or url.username or url.password or url.query or url.fragment:
                raise ValueError("Use an HTTPS API base URL, without credentials/query/fragment")
            if not model.strip() or not api_key.strip():
                raise ValueError("Live mode requires a model and secret key")
        self.mode, self.model, self.family = mode, model, family
        self.base_url, self._api_key = base_url.rstrip("/"), api_key
        self.max_calls, self.max_tokens, self.timeout = max_calls, max_tokens, timeout
        self.token_field, self.trace = token_field, []
        self.extra_body = dict(extra_body or {})
        self._freeinference = urllib.parse.urlsplit(self.base_url).hostname == "freeinference.org"
        self._attempts, self._lock = 0, threading.Lock()

    def ask(self, prompt, *, system="You are a concise teaching assistant.", practice=""):
        if not isinstance(prompt, str) or len(prompt) + len(system) > 28000:
            raise ValueError("Prompt must be text and total prompt length <= 28,000 characters")
        with self._lock:
            if self._attempts >= self.max_calls:
                raise RuntimeError("Call budget reached; inspect the trace before increasing it")
            self._attempts += 1
            call_id = self._attempts
        started = time.perf_counter()
        row = dict(call=call_id, mode=self.mode, model=self.model or "scripted-fixture",
                   family=self.family or ("practice" if self.mode == "practice" else None), input_tokens=None, output_tokens=None,
                   cost=None, status="ok")
        try:
            if self.mode == "practice":
                content = practice
            else:
                messages = [{"role": "system", "content": system}, {"role": "user", "content": prompt}]
                body = {"model": self.model, "messages": messages, self.token_field: self.max_tokens}
                body.update(self.extra_body)
                if self._freeinference:
                    body.update({"stream": False, "thinking": {"type": "disabled"}})
                req = urllib.request.Request(self.base_url + "/chat/completions",
                    data=json.dumps(body).encode(), method="POST",
                    headers={"Authorization": "Bearer " + self._api_key, "Content-Type": "application/json"})
                try:
                    with _NETWORK_SLOTS:
                        with urllib.request.build_opener(_NoRedirect).open(req, timeout=self.timeout) as response:
                            payload = json.load(response)
                except urllib.error.HTTPError as exc:
                    if exc.code == 429:
                        raise RuntimeError("Gateway HTTP 429: rate/quota limit. Check your quota; no retry or key rotation made.") from None
                    raise RuntimeError(f"Gateway HTTP {exc.code}; check provider access/configuration. No retry made.") from None
                except (urllib.error.URLError, TimeoutError, OSError, ValueError):
                    raise RuntimeError("Gateway/network/JSON error; inspect provider configuration. No retry made.") from None
                try:
                    choice = payload["choices"][0]
                    if choice.get("finish_reason") == "length":
                        raise RuntimeError("Output truncated; inspect token budget before retrying")
                    content = choice["message"]["content"]
                    if not isinstance(content, str):
                        raise ValueError("Missing text response")
                    usage = payload.get("usage") or {}
                    row["input_tokens"] = usage.get("prompt_tokens")
                    row["output_tokens"] = usage.get("completion_tokens")
                    if self._freeinference and "model is starting up" in content.lower():
                        raise RuntimeError("FreeInference model is starting up. Wait about 120 seconds, then retry manually; this attempt used the call budget.")
                except (KeyError, IndexError, TypeError, ValueError, AttributeError):
                    raise RuntimeError("Gateway response has no usable text; schema/capability check needed") from None
            return content
        except Exception:
            row["status"] = "error"
            raise
        finally:
            row["latency_s"] = round(time.perf_counter() - started, 4)
            with self._lock:
                self.trace.append(row)


def secret(name="FREEINFERENCE_API_KEY"):
    """Read a Colab secret, then fall back to a hidden local prompt. Never print it."""
    try:
        from google.colab import userdata
        key = userdata.get(name)
        if key:
            return key
    except Exception:
        pass
    return getpass.getpass(f"{name} (hidden): ")


def parse_object(text):
    value = json.loads(text)
    if not isinstance(value, dict):
        raise ValueError("Expected one JSON object")
    return value


def summarize_trace(clients):
    rows = [row for client in clients for row in client.trace]
    return dict(calls=len(rows), failed_calls=sum(r["status"] != "ok" for r in rows),
                input_tokens=sum(r["input_tokens"] for r in rows) if rows and all(isinstance(r["input_tokens"], int) for r in rows) else None,
                output_tokens=sum(r["output_tokens"] for r in rows) if rows and all(isinstance(r["output_tokens"], int) for r in rows) else None,
                summed_call_latency_s=round(sum(r["latency_s"] for r in rows), 4), cost=None)

In [ ]:
# Run all examples without a key first. Change only MODE for a live model run.
MODE = "practice"  # "live" uses the API key supplied through Colab Secrets
BASE_URL = "https://freeinference.org/v1"
MODEL = "qwen3.6-35b"
SECRET_NAME = "FREEINFERENCE_API_KEY"

client = Client(mode=MODE, base_url=BASE_URL, model=MODEL,
                api_key=secret(SECRET_NAME) if MODE == "live" else "",
                max_calls=24, max_tokens=512)
print("Mode:", MODE)

## One pattern at a time

**Goal:** recognize routing, chaining, parallelization, evaluation and orchestration. Each example below is independent and small. We reuse simple support messages, not a large agent framework.

The application chooses the control flow in these workflows. A model may fill in a route or plan, but Python still checks it.

## 1. Routing: choose a branch

A router classifies a message, then code chooses a response path.

In [ ]:
message = "My delivery has not arrived."
route = client.ask('Return only delivery or account for: ' + message,
                   practice="delivery").strip().lower()
replies = {"delivery": "Please share the tracking number.",
           "account": "Please share the login error message."}
if route not in replies:
    raise ValueError("Unknown route: ask a human")
print(replies[route])
assert route in replies

**Expected practice output:** “Please share the tracking number.”

**Tiny exercise:** change the message to a login problem. What should happen when the model returns an unknown label?

## 2. Chaining: use one output as the next input

The first call identifies the customer's need. The second call writes a reply using that result. A bad first step can affect the second.

In [ ]:
need = client.ask("Name the missing detail in: 'My delivery is late.'",
                  practice="tracking number")
reply = client.ask("Write one polite question asking only for: " + need,
                   practice="Could you share the tracking number?")
print("Step 1:", need)
print("Step 2:", reply)
assert need.strip() and reply.strip()

**Tiny exercise:** add a check that `need` is shorter than 80 characters before making the second call.

## 3. Parallelization: independent checks

Two checks can run independently: tone and missing details. Their outputs are not a debate, because neither sees the other's answer. The shared client limits concurrent network calls to two.

In [ ]:
from concurrent.futures import ThreadPoolExecutor
checks = [("Describe the tone of: 'Where is my parcel?'", "The tone is direct."),
          ("What detail is missing from: 'Where is my parcel?'", "The tracking number.")]
def check(item):
    prompt, fixture = item
    return client.ask(prompt, practice=fixture)
with ThreadPoolExecutor(max_workers=2) as pool:
    results = list(pool.map(check, checks))
print(results)
assert len(results) == 2

**Tiny exercise:** compare this with a sequential `for` loop. Parallel execution can reduce wall time, but it does not reduce the number of calls.

## 4. Evaluator–optimizer: critique once, then revise

Use a specific criterion and a fixed stopping point. Critique may help, but a model can also repeat or introduce a mistake.

In [ ]:
draft = "We will sort it out."
critique = client.ask("Does this reply ask for a tracking number? Explain briefly: " + draft,
                      practice="No. Ask for the tracking number instead of promising a result.")
revision = client.ask("Revise once using this feedback: " + critique + "\nDraft: " + draft,
                      practice="Could you share your tracking number so we can investigate?")
print("Feedback:", critique)
print("Revision:", revision)
assert revision.strip()

**Tiny exercise:** give an unhelpful critique. Does the revision become worse? Keep both outputs instead of assuming the latest one is better.

## 5. Optional extension: orchestrator–workers

An orchestrator proposes a small task list. We validate the plan against allowed worker jobs, then run those jobs. The model cannot invent a new tool or arbitrary executable step.

In [ ]:
plan = json.loads(client.ask('Plan checks for "Where is my parcel?". Return JSON '
    '{"tasks":[...]}, using only tone and missing_detail, at most two tasks.',
    practice='{"tasks":["tone","missing_detail"]}'))
workers = {"tone": "Describe the tone.", "missing_detail": "Name the missing detail."}
assert set(plan) == {"tasks"} and isinstance(plan["tasks"], list)
assert 1 <= len(plan["tasks"]) <= 2
assert all(isinstance(task, str) and task in workers for task in plan["tasks"])
worker_results = [client.ask(workers[task] + ' Message: "Where is my parcel?"',
    practice={"tone": "Direct.", "missing_detail": "Tracking number."}[task])
    for task in plan["tasks"]]
print(dict(zip(plan["tasks"], worker_results)))

**Tiny exercise:** replace one planned task with `send_email`. Confirm validation rejects it before any worker executes. The example only writes suggestions; it sends nothing.

**Compare the patterns:** routing chooses one branch; chaining passes an output forward; parallelization gathers independent outputs; evaluation revises a draft; an orchestrator chooses subtasks. Use the smallest pattern that addresses the need, and compare quality, calls and latency.

## Read further

- [Anthropic: workflow patterns](https://www.anthropic.com/research/building-effective-agents)

### Finish
Explain the result in your own words. Make the small change suggested above and rerun its check. If you use a live model, save the actual result and trace; a different or failed response is something to investigate.